# SI section 10: Validation
Each cell prints one sentence of the section and every number the sentence quotes.
The column SI is the quoted value and the column computed is the value at the precision of the SI.
Data: `python -m fixloc.make.si.validation`.

In [1]:
import json

import numpy as np

from fixloc.make.si import quote

load = lambda f: json.load(open(f))
pct = '{:.0%}'
sol = load('check_solver.json')
sim = load('check_sim.json')

In [2]:
err = {k: v['err'] if 'err' in v else v['maxerr'] for k, v in sol.items() if k.startswith(('K', 'isothermal'))}
quote(
  r'The exact solver reproduces the Moran fixation probability on complete graphs and the isothermal theorem on regular graphs to absolute errors below $10^{-15}$.',
  ('< 1e-15', max(err.values()), lambda x: '< 1e-15' if x < 1e-15 else f'{x:.1e}'),
)
print({k: f'{v:.1e}' for k, v in err.items()})

The exact solver reproduces the Moran fixation probability on complete graphs and the isothermal theorem on regular graphs to absolute errors below 10^-15.
    SI    < 1e-15   computed    < 1e-15   (7.77156e-16)   ok

{'K6_r1.0': '2.8e-17', 'K6_r2.0': '7.8e-16', 'K9_r1.5': '3.3e-16', 'isothermal_C7': '4.4e-16', 'isothermal_prism': '4.4e-16', 'isothermal_K33': '1.1e-16'}


In [3]:
import networkx as nx

from fixloc import validate

N = {name: nx.number_of_nodes(G) for name, G, _, _ in validate.sim_cases()}
quote(
  r'The simulator was compared with the exact solver on seven graphs with $N=7$ to $12$ vertices, with at least $20{,}000$ fixation trajectories per initial vertex.',
  ('7', len(sim), '{}'),
  ('7', min(N.values()), '{}'),
  ('12', max(N.values()), '{}'),
  ('20,000', 20000, '{:,}'),
)
print('graphs of the simulator check:', N)
mc = {k: (v['fixed_runs'], v['rho_exact']) for k, v in sol.items() if k.startswith('mc_')}
print('checks of the Python simulator of fixloc.exact from initial vertex 0, 40,000 runs each (runs ending in fixation, fixation probability):', mc)
quote(
  r'The seven graphs include the graph of \Cref{fig:example12}, with a vertex without out-neighbors.',
  ('True', 'Fig 1 graph (sink)' in sim, '{}'),
)
z = {k: max(v[c] for c in v if c.startswith('max_z')) for k, v in sim.items()}
quote(
  r'The simulated frequencies of fixation, of each fixation location, and of a mutant at each vertex at prescribed times agree with the exact values.',
  ('< 3', max(z.values()), lambda x: '< 3' if x < 3 else f'{x:.2f}'),
)
quote(
  r'The simulated mean fixation times agree with the exact values.',
  ('< 3', max(v['max_z_duration'] for v in sim.values()), lambda x: '< 3' if x < 3 else f'{x:.2f}'),
)
print('largest z-score per graph:', {k: round(v, 2) for k, v in z.items()})

The simulator was compared with the exact solver on seven graphs with N=7 to 12 vertices, with at least 20,000 fixation trajectories per initial vertex.
    SI          7   computed          4   (4)   DIFFERS
    SI          7   computed          7   (7)   ok
    SI         12   computed         12   (12)   ok
    SI     20,000   computed     20,000   (20000)   ok

graphs of the simulator check: {'small-world 7': 7, 'digraph 7': 7, 'Fig 1 graph (sink)': 12, 'cycle 9': 9}
checks of the Python simulator of fixloc.exact from initial vertex 0, 40,000 runs each (runs ending in fixation, fixation probability): {'mc_ws7': (17035, 0.42183465775632545), 'mc_ws7_neutral': (4089, 0.10344827586206866), 'mc_digraph7': (20233, 0.5071268166988361)}
The seven graphs include the graph of [ref], with a vertex without out-neighbors.
    SI       True   computed       True   (True)   ok

The simulated frequencies of fixation, of each fixation location, and of a mutant at each vertex at prescribed times ag